# 13 · Análisis de resultados

Genera todas las figuras y tablas del informe a partir del CSV del barrido. No entrena nada: es puramente derivado, así que puede reejecutarse en segundos.

**Entradas**

- `results/metricas/barrido.csv`
- `results/metricas/baseline.csv`
- `models/generadores/*/historial.csv`

**Salidas**

- `results/figures/*.png`
- `results/metricas/resumen_informe.csv`

**Tiempo estimado:** ~3 min en CPU.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import evaluacion
from src.generadores.base import REGISTRO

tabla = evaluacion.cargar_metricas("barrido")
tabla["n_reales"] = tabla["n_reales"].astype(str)

por_tarea = {t: tabla[tabla["tarea"] == t] for t in tabla["tarea"].unique()}
print({t: len(d) for t, d in por_tarea.items()})
tabla.head()

## Métricas de referencia por tarea

En régimen las clases están muy desbalanceadas, así que el accuracy es engañoso: un
modelo que nunca prediga crisis acierta el 90 % y es inútil. Las métricas que se
reportan son el F1 macro y el **recall de crisis**, que es la magnitud que decide si
el trabajo tiene sentido. Si añadir sintéticos de crisis no sube el recall de
crisis, la hipótesis del taller no se sostiene por bien que se comporte el resto.

En volatilidad se reportan MAE y QLIKE. QLIKE penaliza de forma asimétrica y castiga
más infraestimar el riesgo que sobreestimarlo, que es la asimetría correcta en
gestión de riesgo.

In [ ]:
METRICAS = {
    "regimen": ["f1_macro", "recall_crisis"],
    "volatilidad": ["mae", "qlike"],
}
RATIOS = sorted(tabla.loc[tabla["ratio"] > 0, "ratio"].unique())
NIVELES = ["250", "500", "1000", "2000", "todos"]
POLITICA = "equilibrado"

print("ratios:", RATIOS)
print("niveles de reales:", NIVELES)

## Figura principal: métrica frente al volumen de datos reales

Es el gráfico que responde a la pregunta del taller. Los sintéticos deben aportar
cuando hay pocos datos reales y dejar de aportar cuando ya hay muchos, de modo que
las líneas de colores han de separarse de la gris por la izquierda y confluir con
ella por la derecha.

El eje horizontal es logarítmico porque los niveles de reales crecen
geométricamente. La línea gris discontinua es la referencia sin sintéticos: no es
una serie más que comparar, es el suelo que hay que batir.

In [ ]:
for tarea, metricas in METRICAS.items():
    datos = por_tarea[tarea]
    for metrica in metricas:
        for ratio in RATIOS:
            fig, eje = plt.subplots(figsize=(9, 5))
            viz.metrica_vs_reales(datos, metrica, ratio=ratio, politica=POLITICA, eje=eje)
            eje.set_title("{} · {} · ratio {:g}× · {}".format(
                tarea, metrica.replace("_", " "), ratio, POLITICA))
            viz.guardar(fig, "vs_reales_{}_{}_r{}".format(
                tarea, metrica, str(ratio).replace(".", "p")))
            plt.close(fig)

print("Figuras 'métrica frente a reales' generadas:",
      sum(len(m) for m in METRICAS.values()) * len(RATIOS))

## Métrica frente a la proporción de sintéticos

Con los reales fijos, cuánto sintético conviene añadir. La lectura interesante es la
forma: si la métrica sube y luego baja, hay un óptimo intermedio y meter sintéticos
sin límite es contraproducente. La horizontal discontinua marca el resultado sin
ningún sintético en ese mismo nivel de reales.

In [ ]:
for tarea, metricas in METRICAS.items():
    datos = por_tarea[tarea]
    for metrica in metricas:
        for nivel in NIVELES:
            if nivel not in set(datos["n_reales"]):
                continue
            fig, eje = plt.subplots(figsize=(9, 5))
            viz.metrica_vs_ratio(datos, metrica, n_reales=nivel, politica=POLITICA, eje=eje)
            eje.set_title("{} · {} · {} ventanas reales".format(
                tarea, metrica.replace("_", " "), nivel))
            viz.guardar(fig, "vs_ratio_{}_{}_{}".format(tarea, metrica, nivel))
            plt.close(fig)

print("Figuras 'métrica frente a ratio' generadas.")

## Efecto de la política de reparto

Compara `proporcional` contra `equilibrado` en el recall de crisis. Es el contraste
que aísla la hipótesis: si concentrar los sintéticos en la clase rara no bate a
replicar el desbalance, el beneficio observado es regularización genérica y no
cobertura de la clase de crisis.

In [ ]:
datos = por_tarea["regimen"]
fig, ejes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for eje, politica in zip(ejes, ["proporcional", "equilibrado"]):
    viz.metrica_vs_reales(datos, "recall_crisis", ratio=2.0, politica=politica, eje=eje)
    eje.set_title("recall de crisis · ratio 2× · " + politica)

fig.tight_layout()
viz.guardar(fig, "politicas_recall_crisis")

## Comparación con reponderar la pérdida

Los pesos por clase son la alternativa gratuita al desbalance. Esta comparación es
obligada: si reponderar iguala al mejor generador, los sintéticos no aportan nada que
no se consiguiera sin ellos, y el informe debe decirlo.

In [ ]:
baseline = evaluacion.cargar_metricas("baseline").set_index("variante")

mejor = (
    por_tarea["regimen"]
    .query("n_reales == 'todos' and ratio > 0")
    .sort_values("recall_crisis", ascending=False)
    .head(5)[["generador", "ratio", "politica", "f1_macro", "recall_crisis"]]
)

comparativa = pd.DataFrame({
    "sin sintéticos": baseline.loc["regimen", ["f1_macro", "recall_crisis"]],
    "pesos por clase": baseline.loc["regimen_pesos", ["f1_macro", "recall_crisis"]],
    "mejor generador": mejor.iloc[0][["f1_macro", "recall_crisis"]],
})
display(comparativa.round(4))
mejor.round(4)

## Convergencia de los siete generadores

El enunciado exige curvas de loss por entrenamiento. Este panel las reúne todas
desde `models/generadores/*/historial.csv`, sin reentrenar nada.

Los generadores analíticos (jitter, gaussiano, RBIG) no tienen pérdida y reportan su
diagnóstico equivalente; en la cGAN se marca el equilibrio log 2, que es la única
referencia con la que su curva es legible.

In [ ]:
historiales = {}
for nombre in sorted(REGISTRO):
    ruta = src.DIR_MODELOS_GEN / nombre / "historial.csv"
    if ruta.exists():
        historiales[nombre] = pd.read_csv(ruta)

n = len(historiales)
filas_panel = (n + 1) // 2
fig, ejes = plt.subplots(filas_panel, 2, figsize=(13, 3.4 * filas_panel))
ejes_planos = np.ravel(ejes)

for eje, (nombre, historial) in zip(ejes_planos, historiales.items()):
    viz.curva_convergencia(
        historial.drop(columns="regimen", errors="ignore"),
        nombre.replace("_", " "),
        referencia=0.693 if nombre == "cgan" else None,
        eje=eje,
    )
for eje in ejes_planos[n:]:
    eje.set_visible(False)

fig.tight_layout()
viz.guardar(fig, "convergencia_todos")
print("Historiales encontrados:", list(historiales))

## Tabla del informe

Una fila por generador con su mejor resultado en el régimen de escasez (250 y 500
ventanas reales), que es donde la hipótesis predice el efecto, y su resultado con
todos los reales, que es donde debería desvanecerse. La columna de referencia es la
misma métrica sin sintéticos.

In [ ]:
def resumen_tarea(datos, metrica, mayor_mejor=True):
    referencia = (
        datos[datos["generador"] == "solo_real"]
        .set_index("n_reales")[metrica]
    )
    agregado = "max" if mayor_mejor else "min"
    pivote = (
        datos[datos["ratio"] > 0]
        .pivot_table(index="generador", columns="n_reales", values=metrica, aggfunc=agregado)
        .reindex(columns=[n for n in NIVELES if n in referencia.index])
    )
    pivote.loc["solo_real"] = referencia.reindex(pivote.columns)
    return pivote


resumen_regimen = resumen_tarea(por_tarea["regimen"], "recall_crisis", mayor_mejor=True)
resumen_vol = resumen_tarea(por_tarea["volatilidad"], "qlike", mayor_mejor=False)

informe = pd.concat({"recall_crisis": resumen_regimen, "qlike": resumen_vol}, axis=1)
informe.to_csv(src.DIR_METRICAS / "resumen_informe.csv")
informe.round(4)

## Salidas generadas

In [ ]:
# Muestra representativa; el barrido de figuras escribe varias decenas de PNG.
from pathlib import Path

salidas = [
    src.DIR_METRICAS / "resumen_informe.csv",
    src.DIR_FIGURAS / "politicas_recall_crisis.png",
    src.DIR_FIGURAS / "convergencia_todos.png",
    src.DIR_FIGURAS / "vs_reales_regimen_recall_crisis_r1p0.png",
    src.DIR_FIGURAS / "vs_ratio_regimen_recall_crisis_250.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))


In [ ]:
figuras = sorted(p.name for p in src.DIR_FIGURAS.glob("*.png"))
print(len(figuras), "figuras en results/figures/")
figuras